<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_06_TFL_Replay_and_Source_Provenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import os, json, re, hashlib, subprocess, sys, shutil, ast, urllib.request
from datetime import datetime, timezone

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

ROOT = Path("/content")
DRIVE_ROOT = Path("/content/drive/MyDrive/NeuroFHIR_ICHI2027")
PHASE05_DIR = DRIVE_ROOT / "phase05_outputs"
PHASE06_DIR = DRIVE_ROOT / "phase06_outputs"
PHASE06_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSON = PHASE06_DIR / "ICHI2027_Phase06_SHAREABLE.json"
WORK = ROOT / "ichi2027_phase06"
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True, exist_ok=True)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def run(cmd, cwd=None, check=True):
    p = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if check and p.returncode != 0:
        raise RuntimeError(p.stderr[-4000:])
    return p

def now_utc():
    return datetime.now(timezone.utc).isoformat()

Mounted at /content/drive


In [2]:
phase05_candidates = [
    PHASE05_DIR / "ICHI2027_Phase05_SHAREABLE.json",
    ROOT / "ICHI2027_Phase05_SHAREABLE.json",
]
phase05_path = next((p for p in phase05_candidates if p.exists()), None)

if phase05_path is None:
    from google.colab import files
    uploaded = files.upload()
    names = [n for n in uploaded if n.endswith(".json")]
    if not names:
        raise FileNotFoundError("ICHI2027_Phase05_SHAREABLE.json is required")
    phase05_path = ROOT / names[0]
    phase05_path.write_bytes(uploaded[names[0]])

phase05 = json.loads(phase05_path.read_text())
assert phase05.get("phase") == "05_real_provenance_and_tfl_readiness"
phase05_sha = sha256_file(phase05_path)

In [3]:
repos = {
    "ohdsi_tfl": {
        "url": "https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git",
        "commit": "a4b7265adabf096e84b957de068e24a5ab853425",
    },
    "imagewg": {
        "url": "https://github.com/OHDSI/ImageWG.git",
        "commit": "8f0a19bb94e941bb335a097bc9851fab672621c7",
    },
    "pyomop": {
        "url": "https://github.com/dermatologist/pyomop.git",
        "commit": "dfaa6d4062dbc600f52f5377019a347c4caa2cc7",
    },
}

repo_status = {}
for name, cfg in repos.items():
    dest = WORK / name
    run(["git", "clone", "--quiet", cfg["url"], str(dest)])
    run(["git", "checkout", "--quiet", cfg["commit"]], cwd=dest)
    actual = run(["git", "rev-parse", "HEAD"], cwd=dest).stdout.strip()
    repo_status[name] = {
        "expected_commit": cfg["commit"],
        "actual_commit": actual,
        "verified": actual == cfg["commit"],
    }
assert all(x["verified"] for x in repo_status.values())

In [4]:
import nbformat

tfl_repo = WORK / "ohdsi_tfl"
runner = tfl_repo / "FHIRy_pyOMOP_TFL_SubmissionFreezeRunner_v15.ipynb"
assert runner.exists()

nb = nbformat.read(runner, as_version=4)
runner_sha = sha256_file(runner)
code = [c for c in nb.cells if c.cell_type == "code"]

path_literals = []
for i, c in enumerate(code):
    try:
        tree = ast.parse(c.source)
    except Exception:
        continue
    for node in ast.walk(tree):
        if isinstance(node, ast.Constant) and isinstance(node.value, str):
            s = node.value.strip()
            if re.search(r"\.(csv|parquet|json|zip|ipynb|xlsx|txt)$", s, flags=re.I):
                path_literals.append({"cell_index": i, "literal": s})

referenced_basenames = sorted(set(Path(x["literal"].replace("\\","/")).name for x in path_literals))
repo_files = {}
for p in tfl_repo.rglob("*"):
    if p.is_file():
        repo_files.setdefault(p.name, []).append(str(p.relative_to(tfl_repo)))

dependency_inventory = [{
    "basename": b,
    "found_in_repo": b in repo_files,
    "repo_matches": repo_files.get(b, [])
} for b in referenced_basenames]
missing_refs = [x["basename"] for x in dependency_inventory if not x["found_in_repo"]]

/usr/local/lib/python3.13/dist-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


In [5]:
run([sys.executable, "-m", "pip", "install", "-q", "nbclient", "jupyter_client"])
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

execution_dir = WORK / "tfl_v15_execution"
execution_dir.mkdir(parents=True, exist_ok=True)
for item in tfl_repo.iterdir():
    target = execution_dir / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

nb_exec = nbformat.read(execution_dir / runner.name, as_version=4)
execution = {"attempted": True, "completed": False, "error_type": None, "error_message_redacted": None}

try:
    client = NotebookClient(
        nb_exec,
        timeout=1200,
        kernel_name="python3",
        resources={"metadata": {"path": str(execution_dir)}},
        allow_errors=False,
    )
    client.execute()
    execution["completed"] = True
except Exception as e:
    execution["error_type"] = type(e).__name__
    msg = re.sub(r"/content/[^\s'\"`]+", "<REDACTED_PATH>", str(e))
    execution["error_message_redacted"] = msg[-2500:]

executed_path = execution_dir / "FHIRy_pyOMOP_TFL_SubmissionFreezeRunner_v15_EXECUTED.ipynb"
nbformat.write(nb_exec, executed_path)

In [6]:
urls = [
    "https://adni.loni.usc.edu/quick-start-guide-asset/MRI_tables.html",
    "https://adni.loni.usc.edu/data-samples/adni-data/neuroimaging/mri/",
]
terms = [
    "LHIPPOC","RHIPPOC","LENTORHIN","RENTORHIN","LMIDTEMP","RMIDTEMP",
    "VENTRICLES","ICV","FreeSurfer","mm3","mm^3","cubic millimeter"
]

official_doc_evidence = []
for url in urls:
    rec = {"url": url, "status": None, "sha256": None, "matches": []}
    try:
        req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as r:
            raw = r.read()
        text = raw.decode("utf-8", errors="ignore")
        rec["status"] = "RETRIEVED"
        rec["sha256"] = hashlib.sha256(raw).hexdigest()
        low = text.lower()
        for term in terms:
            pos = low.find(term.lower())
            if pos >= 0:
                snippet = re.sub(r"\s+", " ", text[max(0,pos-180):pos+280])
                rec["matches"].append({"term": term, "snippet": snippet[:500]})
    except Exception as e:
        rec["status"] = "ERROR"
        rec["error_type"] = type(e).__name__
    official_doc_evidence.append(rec)

In [7]:
imagewg = WORK / "imagewg"
field_spec = imagewg / "inst/csv/new_csv/OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv"

ddl_candidates = []
for p in imagewg.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".sql",".ddl"}:
        txt = p.read_text(errors="ignore").lower()
        if "image_occurrence" in txt or "image_feature" in txt:
            ddl_candidates.append({
                "path": str(p.relative_to(imagewg)),
                "sha256": sha256_file(p),
                "mentions_image_occurrence": "image_occurrence" in txt,
                "mentions_image_feature": "image_feature" in txt,
            })

micdm = {
    "field_spec_present": field_spec.exists(),
    "field_spec_sha256": sha256_file(field_spec) if field_spec.exists() else None,
    "executable_ddl_candidates": ddl_candidates,
    "executable_ddl_found": len(ddl_candidates) > 0,
}

In [8]:
pyomop = WORK / "pyomop"
pyomop_inventory = {
    "cdm54_measurement_model_present": (pyomop / "src/pyomop/cdm54/cdm54_tables.py").exists(),
    "cdm6_measurement_model_present": (pyomop / "src/pyomop/cdm6/cdm6_tables.py").exists(),
    "loader_present": (pyomop / "src/pyomop/loader.py").exists(),
    "migration_mapping_present": (pyomop / "src/pyomop/migrate/pyomop_migrate.py").exists(),
}

In [9]:
unit_hits = []
for rec in official_doc_evidence:
    found = {m["term"].lower() for m in rec.get("matches", [])}
    has_feature = any(t in found for t in {
        "lhippoc","rhippoc","lentorhin","rentorhin","lmidtemp","rmidtemp","ventricles","icv"
    })
    has_unit = any(t in found for t in {"mm3","mm^3","cubic millimeter"})
    if rec.get("status") == "RETRIEVED" and has_feature and has_unit:
        unit_hits.append(rec["url"])

shareable = {
    "project": "ICHI2027_neuroimaging_fhir_omop_fidelity",
    "phase": "06_original_tfl_replay_and_source_provenance",
    "run_utc": now_utc(),
    "scientific_results_status": "METHOD_REPLAY_AND_PROVENANCE_EVIDENCE_ONLY_NO_FOUR_ARM_ETL_COMPARISON",
    "privacy": "Aggregate repository, execution, and public-document evidence only; no participant identifiers or source rows.",
    "phase05_sha256": phase05_sha,
    "pinned_repositories": repo_status,
    "tfl_v15_runner": {
        "sha256": runner_sha,
        "code_cells": len(code),
        "dependency_inventory": dependency_inventory,
        "missing_referenced_basenames_from_repo": missing_refs,
        "execution": execution,
    },
    "official_adni_public_document_evidence": official_doc_evidence,
    "authoritative_unit_evidence_urls": unit_hits,
    "micdm": micdm,
    "pyomop_inventory": pyomop_inventory,
    "gates": {
        "original_tfl_v15_execution_completed": execution["completed"],
        "authoritative_unit_evidence_found_for_exact_local_export": bool(unit_hits),
        "dicom_study_uid_provenance_verified": False,
        "dicom_series_uid_provenance_verified": False,
        "omop_vocabulary_snapshot_pinned": False,
        "micdm_executable_ddl_found": micdm["executable_ddl_found"],
        "ready_for_real_micdm_load": False,
        "ready_for_four_arm_comparison": False,
    },
    "actual_four_arm_comparator_scores": None,
    "real_micdm_load_results": None,
}

if not execution["completed"]:
    shareable["next_stage_gate"] = "RESOLVE_ORIGINAL_TFL_V15_RUNTIME_DEPENDENCIES_AND_REPLAY_BEFORE_COMPARATOR_ADAPTER"
elif not bool(unit_hits):
    shareable["next_stage_gate"] = "VERIFY_EXACT_ADNI_EXPORT_UNITS_AND_PROCESSING_LINEAGE_BEFORE_REAL_MICDM_LOAD"
elif not micdm["executable_ddl_found"]:
    shareable["next_stage_gate"] = "PIN_OR_BUILD_EXACT_MICDM_TARGET_SCHEMA_FROM_VERIFIED_FIELD_SPEC_AND_PIN_OMOP_VOCAB"
else:
    shareable["next_stage_gate"] = "PIN_OMOP_VOCAB_AND_BUILD_AUDITED_REAL_TARGET_ADAPTER"

OUT_JSON.write_text(json.dumps(shareable, indent=2))
print(OUT_JSON)

/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase06_outputs/ICHI2027_Phase06_SHAREABLE.json


In [10]:
try:
    from google.colab import files
    files.download(str(OUT_JSON))
except Exception:
    print(OUT_JSON.read_text()[:4000])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>